# Pipeline Bronze : JSON → Delta Lake

## Objectif
Implémenter une pipeline permettant de :
1. Lire un flux de données au format JSON
2. Appliquer des transformations basiques (nettoyage, filtrage, projection)
3. Écrire les données dans une table Delta (niveau Bronze)
4. Configurer une stratégie de tolérance aux pannes via checkpointing

## Contexte SmartTech
Les données proviennent de capteurs IoT installés dans des bâtiments intelligents :
- Température, humidité, consommation d'énergie
- Détection d'anomalies
- Informations de localisation


## 1. Configuration de l'environnement Spark


In [ ]:
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *
import os

# Les variables d'environnement sont injectées par docker-compose depuis le fichier .env
# Pas besoin de load_dotenv() car docker-compose passe les variables directement au conteneur

# Configuration des chemins depuis les variables d'environnement (avec valeurs par défaut)
DATA_DIR = os.getenv("DATA_DIR", "/opt/spark/data")
DELTA_BRONZE_PATH = os.getenv("DELTA_BRONZE_PATH", "/opt/spark/delta/bronze")
CHECKPOINT_PATH = os.getenv("CHECKPOINT_BRONZE_PATH", "/opt/spark/checkpoints/bronze")

# Configuration Spark depuis les variables d'environnement
SPARK_APP_NAME = os.getenv("SPARK_APP_NAME", "SmartTech-Bronze-Pipeline")

# Créer la session Spark avec support Delta Lake
# IMPORTANT : delta-spark==2.4.0 est installé via pip dans le Dockerfile
# configure_spark_with_delta_pip() utilise les JARs déjà installés par pip (pas de téléchargement Maven)
builder = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")

# Utiliser configure_spark_with_delta_pip() qui utilise les JARs installés par pip (delta-spark==2.4.0)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("✓ Spark Session créée avec succès")
print(f"✓ Version Spark : {spark.version}")
print(f"✓ DATA_DIR : {DATA_DIR}")
print(f"✓ DELTA_BRONZE_PATH : {DELTA_BRONZE_PATH}")
print(f"✓ CHECKPOINT_PATH : {CHECKPOINT_PATH}")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b94e2181-cff1-408a-ab88-c90cbf57e4f2;1.0
	confs: [default]
:: resolution report :: resolve 2344ms :: artifacts dl 0ms
	:: modules in use:
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   0   |   0   |
	---------------------------------------------------------------------

:: problems summary ::
:::: WARNINGS
		module not found: io.delta#delta-spark_2.13;2.4.0

	==== local-m2-cache: tried

	  file:/root/.m2/repository/io/delta/delta-spark_2.13/2.4.0/delta-spark_2.13-2.4.0.pom

	  -- artifact 

RuntimeError: Java gateway process exited before sending its port number

## 2. Définition du schéma des données


In [ ]:
# Schéma des données IoT SmartTech
sensor_schema = StructType([
    StructField("sensor_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("temperature", DoubleType(), True),
    StructField("humidity", DoubleType(), True),
    StructField("energy_consumption", DoubleType(), True),
    StructField("anomaly_detected", BooleanType(), True),
    StructField("building_id", StringType(), True),
    StructField("sensor_type", StringType(), True),
    StructField("location", StringType(), True)
])

print("✓ Schéma défini pour les données IoT")


## 3. Lecture du flux JSON


In [ ]:
# Configuration de la source de lecture JSON
# maxFilesPerTrigger : nombre de fichiers traités par micro-batch
raw_stream = spark \
    .readStream \
    .format("json") \
    .schema(sensor_schema) \
    .option("maxFilesPerTrigger", 1) \
    .option("multiline", "true") \
    .load(DATA_DIR)

print("✓ Source de lecture JSON configurée")
print(f"✓ Schéma de la source :")
raw_stream.printSchema()


## 4. Transformations basiques

### 4.1 Nettoyage des données


In [ ]:
# Nettoyage et validation des données
cleaned_stream = raw_stream \
    .withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("temperature", 
        when(col("temperature").isNull(), 20.0).otherwise(col("temperature"))
    ) \
    .withColumn("humidity", 
        when(col("humidity").isNull(), 50.0).otherwise(col("humidity"))
    ) \
    .withColumn("energy_consumption", 
        when(col("energy_consumption").isNull(), 0.0).otherwise(col("energy_consumption"))
    )

print("✓ Nettoyage des valeurs nulles effectué")


### 4.2 Filtrage des données invalides


In [ ]:
# Filtrage : garder uniquement les données valides
# Validation des plages de valeurs réalistes
filtered_stream = cleaned_stream \
    .filter(
        (col("sensor_id").isNotNull()) &
        (col("timestamp").isNotNull()) &
        (col("temperature") >= -50) & (col("temperature") <= 60) &  # Plage réaliste
        (col("humidity") >= 0) & (col("humidity") <= 100) &  # Humidité en pourcentage
        (col("energy_consumption") >= 0)  # Consommation positive
    )

print("✓ Filtrage des données invalides configuré")


### 4.3 Projection et ajout de colonnes calculées


In [ ]:
# Projection : sélectionner et enrichir les colonnes
bronze_stream = filtered_stream \
    .select(
        col("sensor_id"),
        col("timestamp"),
        col("temperature"),
        col("humidity"),
        col("energy_consumption"),
        col("anomaly_detected"),
        col("building_id"),
        col("sensor_type"),
        col("location"),
        col("ingestion_timestamp"),
        # Colonnes calculées
        when(col("temperature") > 30, "HIGH").otherwise("NORMAL").alias("temp_status"),
        when(col("energy_consumption") > 800, "HIGH").otherwise("NORMAL").alias("energy_status")
    )

print("✓ Projection et enrichissement des données effectués")
print(f"✓ Schéma final Bronze :")
bronze_stream.printSchema()


## 5. Écriture dans Delta Lake (Niveau Bronze)


In [ ]:
# Configuration de l'écriture vers Delta Lake
# Mode append : ajoute uniquement les nouvelles lignes
query = bronze_stream \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("path", DELTA_BRONZE_PATH) \
    .partitionBy("building_id", "sensor_type") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✓ Pipeline Bronze démarrée")
print(f"✓ Écriture dans : {DELTA_BRONZE_PATH}")
print(f"✓ Checkpoint dans : {CHECKPOINT_PATH}")
print(f"✓ Trigger : toutes les 10 secondes")


## 6. Monitoring de la pipeline


In [ ]:
# Attendre quelques micro-batches pour voir les données
import time

print("Pipeline en cours d'exécution...")
print("Attente de 30 secondes pour traiter les données...")

time.sleep(30)

# Vérifier le statut
print(f"\n✓ Statut de la query : {query.status}")
print(f"✓ Dernière progression : {query.lastProgress}")


## 7. Vérification des données écrites


In [ ]:
# Lire les données Delta Lake pour vérification
bronze_df = spark.read.format("delta").load(DELTA_BRONZE_PATH)

print(f"✓ Nombre total d'enregistrements dans Bronze : {bronze_df.count()}")
print("\n✓ Aperçu des données :")
bronze_df.show(10, truncate=False)

print("\n✓ Statistiques par building :")
bronze_df.groupBy("building_id").count().show()

print("\n✓ Statistiques par type de capteur :")
bronze_df.groupBy("sensor_type").count().show()


## 8. Test de tolérance aux pannes


In [ ]:
# Arrêter la query pour simuler une panne
query.stop()
print("✓ Pipeline arrêtée (simulation de panne)")

# Relancer la pipeline - elle devrait reprendre depuis le checkpoint
print("\n✓ Redémarrage de la pipeline depuis le checkpoint...")

query_restart = bronze_stream \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .option("path", DELTA_BRONZE_PATH) \
    .partitionBy("building_id", "sensor_type") \
    .trigger(processingTime="10 seconds") \
    .start()

print("✓ Pipeline redémarrée - les données seront reprises depuis le dernier offset traité")


## 9. Arrêt propre de la pipeline


In [ ]:
# Arrêter la pipeline proprement
query_restart.stop()
print("✓ Pipeline arrêtée proprement")

# Afficher un résumé final
final_count = spark.read.format("delta").load(DELTA_BRONZE_PATH).count()
print(f"\n✓ Total d'enregistrements dans la table Bronze : {final_count}")
print("\n✓ Pipeline Bronze terminée avec succès !")


## Résumé de la pipeline Bronze

### Ce qui a été implémenté :

1. ✅ **Lecture du flux JSON** : Configuration de la source avec schéma défini
2. ✅ **Transformations** :
   - Nettoyage des valeurs nulles
   - Validation des plages de valeurs
   - Filtrage des données invalides
   - Enrichissement avec colonnes calculées
3. ✅ **Écriture Delta Lake** : Niveau Bronze avec partitionnement
4. ✅ **Checkpointing** : Tolérance aux pannes configurée
5. ✅ **Monitoring** : Suivi de l'exécution et vérification des données

### Points clés :
- **Mode append** : Ajoute uniquement les nouvelles lignes
- **Partitionnement** : Par `building_id` et `sensor_type` pour optimiser les requêtes
- **Trigger** : ProcessingTime de 10 secondes
- **Checkpoint** : Permet la reprise après panne sans perte de données
